In [2]:
import os
import glob
import numpy as np
import pandas as pd

# =================================================================
# 0. 自动寻址与加载工具
# =================================================================
def find_csv(base_path):
    files = glob.glob(os.path.join(base_path, "**", "test_dist.csv"), recursive=True)
    if not files:
        print(f"❌ 警告: 找不到文件 {base_path}")
        return None
    return files[0]

BASE_DIR = "/NAS/shith/uplift/results/criteo"

paths = {
    "C": find_csv(f"{BASE_DIR}/train_c/TARNET/c_v1_base"),
    "V1": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v1_base"),
    "V6": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v6_res_moe"),
    "V8_S4": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v8_s4"),
    "V8_S1": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v8_s1_t10")
}

print("📦 [1/4] 加载模型数据...")
dfs = {name: pd.read_csv(path) for name, path in paths.items() if path}

# =================================================================
# 1. 构建全局特征大表
# =================================================================
print("🧩 [2/4] 构建特征与排位库...")
df = pd.DataFrame({
    't': dfs["C"]['t'],
    'y_true': dfs["C"]['y_true'],
    'c_true': dfs["C"]['c_true'],
    
    # 原始 C 模型的先验权重 (即你的 Baseline Weights)
    'C_p_always': dfs["C"]['y0_prob'],
    'C_p_comp': (dfs["C"]['y1_prob'] - dfs["C"]['y0_prob']).clip(lower=0),
    'C_p_never': (1.0 - dfs["C"]['y1_prob']).clip(lower=0)
})

# 各模型最终打分
df['v1_score'] = dfs["V1"]['uplift_pred']
if "V6" in dfs: df['v6_score'] = dfs["V6"]['uplift_pred']
if "V8_S4" in dfs: df['v8s4_score'] = dfs["V8_S4"]['uplift_pred']
if "V8_S1" in dfs: df['v8s1_score'] = dfs["V8_S1"]['uplift_pred']

# 提取 V8_S4 的最终门控
if "V8_S4" in dfs:
    df['s4_g_co'] = dfs["V8_S4"].get('wb_final_pi_comp', np.nan)
    df['s4_g_at'] = dfs["V8_S4"].get('wb_final_pi_always', np.nan)
    df['s4_g_nt'] = dfs["V8_S4"].get('wb_final_pi_never', np.nan)

# 提取 V8_S1 的最终门控与 Delta
if "V8_S1" in dfs:
    df['s1_g_co'] = dfs["V8_S1"].get('wb_final_pi_comp', np.nan)
    df['s1_g_at'] = dfs["V8_S1"].get('wb_final_pi_always', np.nan)
    df['s1_g_nt'] = dfs["V8_S1"].get('wb_final_pi_never', np.nan)
    df['s1_delta'] = dfs["V8_S1"].get('wb_delta_c', np.nan)

# 计算百分制排位 (0 是最高)
df['c_rank'] = df['C_p_comp'].rank(pct=True, ascending=False) * 100
df['v1_rank'] = df['v1_score'].rank(pct=True, ascending=False) * 100
if "V6" in dfs: df['v6_rank'] = df['v6_score'].rank(pct=True, ascending=False) * 100
if "V8_S4" in dfs: df['v8s4_rank'] = df['v8s4_score'].rank(pct=True, ascending=False) * 100
if "V8_S1" in dfs: df['v8s1_rank'] = df['v8s1_score'].rank(pct=True, ascending=False) * 100

# =================================================================
# 2. 定义分析人群 (冲突区 + C 定义的 Top10% 四象限)
# =================================================================
print("⚔️ [3/4] 圈定分析人群...")

th_always = df['C_p_always'].quantile(0.90)
th_comp = df['C_p_comp'].quantile(0.90)
th_never = df['C_p_never'].quantile(0.90)

mask_pure_co = (df['C_p_comp'] >= th_comp) & (df['C_p_always'] < th_always)
mask_pure_at = (df['C_p_always'] >= th_always) & (df['C_p_comp'] < th_comp)
mask_both = (df['C_p_always'] >= th_always) & (df['C_p_comp'] >= th_comp)
mask_nt = (df['C_p_never'] >= th_never) & (df['C_p_always'] < th_always) & (df['C_p_comp'] < th_comp)

mask_conflict_a = (df['v1_rank'] <= 10.0) & (df['c_rank'] > 50.0) # V1陷阱
mask_conflict_b = (df['c_rank'] <= 10.0) & (df['v1_rank'] > 50.0) # C慧眼

groups_to_analyze = [
    ("【冲突区 A】 V1 假阳性陷阱 (V1爱, C恨)", mask_conflict_a),
    ("【冲突区 B】 C 遗漏的金子 (C爱, V1恨)", mask_conflict_b),
    ("【标准集】 Pure CO (金子 Top 10%)", mask_pure_co),
    ("【标准集】 Pure AT (羊毛党 Top 10%)", mask_pure_at),
    ("【标准集】 AT & CO (双料高潜 Top 10%)", mask_both),
    ("【标准集】 NT (绝缘尾部 Top 10%)", mask_nt)
]

# =================================================================
# 3. 辅助函数：计算均值与归一化
# =================================================================
def get_norm_weights(group, prefix):
    if f'{prefix}_co' not in group.columns or group[f'{prefix}_co'].isna().all():
        return None, None
    co = group[f'{prefix}_co'].mean()
    at = group[f'{prefix}_at'].mean()
    nt = group[f'{prefix}_nt'].mean()
    
    total = co + at + nt
    if total > 0:
        n_co, n_at, n_nt = (co/total)*100, (at/total)*100, (nt/total)*100
    else:
        n_co, n_at, n_nt = 0, 0, 0
    return (co, at, nt), (n_co, n_at, n_nt)

# =================================================================
# 4. 打印极其详细的透视报告
# =================================================================
print("📊 [4/4] 生成终极对账单...\n")

for name, mask in groups_to_analyze:
    group = df[mask]
    if len(group) == 0: continue
        
    print("="*110)
    print(f"🎯 {name} | 样本量: {len(group)}")
    print("="*110)
    
    # 1. 最终排位对比
    print("[Uplift 全局排位演进 (数值越小越头部)]")
    v1_r = group['v1_rank'].mean()
    print(f"  盲人底座 (V1) : {v1_r:>6.2f}%")
    if "V6" in dfs: print(f"  残差防守 (V6) : {group['v6_rank'].mean():>6.2f}%")
    if "V8_S4" in dfs: print(f"  特征门控 (S4) : {group['v8s4_rank'].mean():>6.2f}%")
    if "V8_S1" in dfs: print(f"  动态阈值 (S1) : {group['v8s1_rank'].mean():>6.2f}%")
    
    print("\n[门控注意力流向 (Raw 绝对值 -> 归一化比例)]")
    
    # 2. C 先验对照组 (Baseline)
    c_raw, c_norm = get_norm_weights(group.rename(columns={'C_p_comp':'C_p_co', 'C_p_always':'C_p_at', 'C_p_never':'C_p_nt'}), 'C_p')
    print(f"  🔵 C-Prior(基准): [CO:{c_raw[0]:.3f}, AT:{c_raw[1]:.3f}, NT:{c_raw[2]:.3f}]  ==> 占比: (CO: {c_norm[0]:>4.1f}%, AT: {c_norm[1]:>4.1f}%, NT: {c_norm[2]:>4.1f}%)")
    
    # 3. V8-S4 权重
    if "V8_S4" in dfs:
        s4_raw, s4_norm = get_norm_weights(group, 's4_g')
        if s4_raw:
            print(f"  🟢 V8_S4(重配): [CO:{s4_raw[0]:.3f}, AT:{s4_raw[1]:.3f}, NT:{s4_raw[2]:.3f}]  ==> 占比: (CO: {s4_norm[0]:>4.1f}%, AT: {s4_norm[1]:>4.1f}%, NT: {s4_norm[2]:>4.1f}%)")
            
    # 4. V8-S1 权重与 Delta
    if "V8_S1" in dfs:
        s1_raw, s1_norm = get_norm_weights(group, 's1_g')
        if s1_raw:
            delta = group['s1_delta'].mean()
            print(f"  🟠 V8_S1(平移): [CO:{s1_raw[0]:.3f}, AT:{s1_raw[1]:.3f}, NT:{s1_raw[2]:.3f}]  ==> 占比: (CO: {s1_norm[0]:>4.1f}%, AT: {s1_norm[1]:>4.1f}%, NT: {s1_norm[2]:>4.1f}%) | 阈值 Delta: {delta:+.4f}")

    print("-" * 110 + "\n")

print("✅ 所有人群扫描完毕！把结果砸过来吧！")

📦 [1/4] 加载模型数据...
🧩 [2/4] 构建特征与排位库...
⚔️ [3/4] 圈定分析人群...
📊 [4/4] 生成终极对账单...

🎯 【冲突区 A】 V1 假阳性陷阱 (V1爱, C恨) | 样本量: 42845
[Uplift 全局排位演进 (数值越小越头部)]
  盲人底座 (V1) :   6.60%
  残差防守 (V6) :   7.81%
  特征门控 (S4) :   5.65%
  动态阈值 (S1) :   6.48%

[门控注意力流向 (Raw 绝对值 -> 归一化比例)]
  🔵 C-Prior(基准): [CO:0.000, AT:0.202, NT:0.803]  ==> 占比: (CO:  0.0%, AT: 20.1%, NT: 79.9%)
  🟢 V8_S4(重配): [CO:0.000, AT:0.166, NT:0.210]  ==> 占比: (CO:  0.0%, AT: 44.2%, NT: 55.8%)
  🟠 V8_S1(平移): [CO:0.000, AT:0.202, NT:0.803]  ==> 占比: (CO:  0.0%, AT: 20.1%, NT: 79.9%) | 阈值 Delta: +nan
--------------------------------------------------------------------------------------------------------------

🎯 【冲突区 B】 C 遗漏的金子 (C爱, V1恨) | 样本量: 17384
[Uplift 全局排位演进 (数值越小越头部)]
  盲人底座 (V1) :  87.16%
  残差防守 (V6) :  42.27%
  特征门控 (S4) :  42.67%
  动态阈值 (S1) :  35.62%

[门控注意力流向 (Raw 绝对值 -> 归一化比例)]
  🔵 C-Prior(基准): [CO:0.051, AT:0.059, NT:0.890]  ==> 占比: (CO:  5.1%, AT:  5.9%, NT: 89.0%)
  🟢 V8_S4(重配): [CO:0.030, AT:0.050, NT:0.241]  ==> 占比: (CO:  9.

In [4]:
import os
import glob
import numpy as np
import pandas as pd

# =================================================================
# 0. 自动寻址与加载工具
# =================================================================
def find_csv(base_path):
    files = glob.glob(os.path.join(base_path, "**", "test_dist.csv"), recursive=True)
    if not files:
        print(f"❌ 警告: 找不到文件 {base_path}")
        return None
    return files[0]

BASE_DIR = "/NAS/shith/uplift/results/criteo"

paths = {
    "C": find_csv(f"{BASE_DIR}/train_c/TARNET/c_v1_base"),
    "V1": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v1_base"),
    "V6": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v6_res_moe"),
    "V7": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v7_soft_top5"),  # 🌟 新增 V7
    "V8_S4": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v8_s4"),
    "V8_S1": find_csv(f"{BASE_DIR}/train_y/TARNET/y_v8_s1_t10")
}

print("📦 [1/4] 加载模型数据 (V1 -> V6 -> V7 -> V8)...")
dfs = {name: pd.read_csv(path) for name, path in paths.items() if path}

# =================================================================
# 1. 构建全局特征大表
# =================================================================
print("🧩 [2/4] 构建特征与排位库...")
df = pd.DataFrame({
    't': dfs["C"]['t'],
    'y_true': dfs["C"]['y_true'],
    'c_true': dfs["C"]['c_true'],
    
    # 原始 C 模型的先验权重
    'C_p_always': dfs["C"]['y0_prob'],
    'C_p_comp': (dfs["C"]['y1_prob'] - dfs["C"]['y0_prob']).clip(lower=0),
    'C_p_never': (1.0 - dfs["C"]['y1_prob']).clip(lower=0)
})

# 各模型最终打分
df['v1_score'] = dfs["V1"]['uplift_pred']
if "V6" in dfs: df['v6_score'] = dfs["V6"]['uplift_pred']
if "V7" in dfs: df['v7_score'] = dfs["V7"]['uplift_pred'] # 🌟 新增 V7 预测值
if "V8_S4" in dfs: df['v8s4_score'] = dfs["V8_S4"]['uplift_pred']
if "V8_S1" in dfs: df['v8s1_score'] = dfs["V8_S1"]['uplift_pred']

# 🌟 提取 V7 的全局 Gate 与 Final Pi
if "V7" in dfs:
    df['v7_gate'] = dfs["V7"].get('wb_v7_gate', np.nan)
    df['v7_pi_co'] = dfs["V7"].get('wb_final_pi_comp', np.nan)
    df['v7_pi_at'] = dfs["V7"].get('wb_final_pi_always', np.nan)
    df['v7_pi_nt'] = dfs["V7"].get('wb_final_pi_never', np.nan)

# 提取 V8_S4 的 Final_Pi 和 Pure Gate
if "V8_S4" in dfs:
    df['s4_pi_co'] = dfs["V8_S4"].get('wb_final_pi_comp', np.nan)
    df['s4_pi_at'] = dfs["V8_S4"].get('wb_final_pi_always', np.nan)
    df['s4_pi_nt'] = dfs["V8_S4"].get('wb_final_pi_never', np.nan)
    df['s4_gate_co'] = dfs["V8_S4"].get('wb_gate_comp', np.nan)
    df['s4_gate_at'] = dfs["V8_S4"].get('wb_gate_always', np.nan)
    df['s4_gate_nt'] = dfs["V8_S4"].get('wb_gate_never', np.nan)

# 提取 V8_S1 的 Final_Pi, Pure Gate 和 Delta
if "V8_S1" in dfs:
    df['s1_pi_co'] = dfs["V8_S1"].get('wb_final_pi_comp', np.nan)
    df['s1_pi_at'] = dfs["V8_S1"].get('wb_final_pi_always', np.nan)
    df['s1_pi_nt'] = dfs["V8_S1"].get('wb_final_pi_never', np.nan)
    df['s1_gate_co'] = dfs["V8_S1"].get('wb_gate_comp', np.nan)
    df['s1_gate_at'] = dfs["V8_S1"].get('wb_gate_always', np.nan)
    df['s1_gate_nt'] = dfs["V8_S1"].get('wb_gate_never', np.nan)
    df['s1_delta'] = dfs["V8_S1"].get('wb_delta_c', np.nan)

# 计算百分制排位 (0 是最高)
df['c_rank'] = df['C_p_comp'].rank(pct=True, ascending=False) * 100
df['v1_rank'] = df['v1_score'].rank(pct=True, ascending=False) * 100
if "V6" in dfs: df['v6_rank'] = df['v6_score'].rank(pct=True, ascending=False) * 100
if "V7" in dfs: df['v7_rank'] = df['v7_score'].rank(pct=True, ascending=False) * 100 # 🌟 新增 V7 排位
if "V8_S4" in dfs: df['v8s4_rank'] = df['v8s4_score'].rank(pct=True, ascending=False) * 100
if "V8_S1" in dfs: df['v8s1_rank'] = df['v8s1_score'].rank(pct=True, ascending=False) * 100

# =================================================================
# 2. 定义分析人群
# =================================================================
print("⚔️ [3/4] 圈定分析人群...")

th_always = df['C_p_always'].quantile(0.90)
th_comp = df['C_p_comp'].quantile(0.90)
th_never = df['C_p_never'].quantile(0.90)

mask_pure_co = (df['C_p_comp'] >= th_comp) & (df['C_p_always'] < th_always)
mask_pure_at = (df['C_p_always'] >= th_always) & (df['C_p_comp'] < th_comp)
mask_both = (df['C_p_always'] >= th_always) & (df['C_p_comp'] >= th_comp)
mask_nt = (df['C_p_never'] >= th_never) & (df['C_p_always'] < th_always) & (df['C_p_comp'] < th_comp)

mask_conflict_a = (df['v1_rank'] <= 10.0) & (df['c_rank'] > 50.0) # V1陷阱
mask_conflict_b = (df['c_rank'] <= 10.0) & (df['v1_rank'] > 50.0) # C慧眼

groups_to_analyze = [
    ("【冲突区 A】 V1 假阳性陷阱 (V1爱, C恨)", mask_conflict_a),
    ("【冲突区 B】 C 遗漏的金子 (C爱, V1恨)", mask_conflict_b),
    ("【标准集】 Pure CO (金子 Top 10%)", mask_pure_co),
    ("【标准集】 Pure AT (羊毛党 Top 10%)", mask_pure_at),
    ("【标准集】 AT & CO (双料高潜 Top 10%)", mask_both),
    ("【标准集】 NT (绝缘尾部 Top 10%)", mask_nt)
]

# =================================================================
# 3. 辅助函数：计算均值与归一化
# =================================================================
def get_norm_weights(group, prefix):
    if f'{prefix}_co' not in group.columns or group[f'{prefix}_co'].isna().all():
        return None, None
    co = group[f'{prefix}_co'].mean()
    at = group[f'{prefix}_at'].mean()
    nt = group[f'{prefix}_nt'].mean()
    
    total = co + at + nt
    if total > 0:
        n_co, n_at, n_nt = (co/total)*100, (at/total)*100, (nt/total)*100
    else:
        n_co, n_at, n_nt = 0, 0, 0
    return (co, at, nt), (n_co, n_at, n_nt)

# =================================================================
# 4. 打印报告
# =================================================================
print("📊 [4/4] 生成终极对账单...\n")

for name, mask in groups_to_analyze:
    group = df[mask]
    if len(group) == 0: continue
        
    print("="*120)
    print(f"🎯 {name} | 样本量: {len(group)}")
    print("="*120)
    
    # 1. 最终排位对比
    print("[Uplift 全局排位演进 (数值越小越头部)]")
    print(f"  盲人底座 (V1) : {group['v1_rank'].mean():>6.2f}%")
    if "V6" in dfs: print(f"  残差防守 (V6) : {group['v6_rank'].mean():>6.2f}%")
    if "V7" in dfs: print(f"  全局衰减 (V7) : {group['v7_rank'].mean():>6.2f}%") # 🌟
    if "V8_S4" in dfs: print(f"  特征门控 (S4) : {group['v8s4_rank'].mean():>6.2f}%")
    if "V8_S1" in dfs: print(f"  动态阈值 (S1) : {group['v8s1_rank'].mean():>6.2f}%")
    
    print("\n[门控流向解剖 (上帝先验 -> V7衰减 -> V8重配)]")
    
    # C 先验对照组
    c_raw, c_norm = get_norm_weights(group.rename(columns={'C_p_comp':'C_p_co', 'C_p_always':'C_p_at', 'C_p_never':'C_p_nt'}), 'C_p')
    print(f"  🔵 C-Prior先验:   [CO:{c_raw[0]:.3f}, AT:{c_raw[1]:.3f}, NT:{c_raw[2]:.3f}]  ==> 占比: (CO: {c_norm[0]:>4.1f}%, AT: {c_norm[1]:>4.1f}%, NT: {c_norm[2]:>4.1f}%)")
    
    # V7 (全局门控衰减)
    if "V7" in dfs:
        v7_pi_raw, v7_pi_norm = get_norm_weights(group, 'v7_pi')
        if v7_pi_raw:
            v7_gate_mean = group['v7_gate'].mean()
            print(f"  🟡 V7 全局衰减:   [CO:{v7_pi_raw[0]:.3f}, AT:{v7_pi_raw[1]:.3f}, NT:{v7_pi_raw[2]:.3f}]  ==> 占比: (CO: {v7_pi_norm[0]:>4.1f}%, AT: {v7_pi_norm[1]:>4.1f}%, NT: {v7_pi_norm[2]:>4.1f}%) | 统一 Gate: {v7_gate_mean:.4f}")

    # V8-S4
    if "V8_S4" in dfs:
        gate_raw, gate_norm = get_norm_weights(group, 's4_gate')
        pi_raw, pi_norm = get_norm_weights(group, 's4_pi')
        if gate_raw and pi_raw:
            print(f"  🟢 V8_S4纯门控:   [CO:{gate_raw[0]:.3f}, AT:{gate_raw[1]:.3f}, NT:{gate_raw[2]:.3f}]  <-- 网络独立计算方向")
            print(f"  🟢 V8_S4最终Pi:   [CO:{pi_raw[0]:.3f}, AT:{pi_raw[1]:.3f}, NT:{pi_raw[2]:.3f}]  ==> 占比: (CO: {pi_norm[0]:>4.1f}%, AT: {pi_norm[1]:>4.1f}%, NT: {pi_norm[2]:>4.1f}%)")
            
    # V8-S1
    if "V8_S1" in dfs:
        gate_raw, gate_norm = get_norm_weights(group, 's1_gate')
        pi_raw, pi_norm = get_norm_weights(group, 's1_pi')
        if gate_raw and pi_raw:
            delta = group['s1_delta'].mean()
            print(f"  🟠 V8_S1纯门控:   [CO:{gate_raw[0]:.3f}, AT:{gate_raw[1]:.3f}, NT:{gate_raw[2]:.3f}]  <-- 网络平移阈值")
            print(f"  🟠 V8_S1最终Pi:   [CO:{pi_raw[0]:.3f}, AT:{pi_raw[1]:.3f}, NT:{pi_raw[2]:.3f}]  ==> 占比: (CO: {pi_norm[0]:>4.1f}%, AT: {pi_norm[1]:>4.1f}%, NT: {pi_norm[2]:>4.1f}%) | Delta: {delta:+.4f}")

    print("-" * 120 + "\n")

print("✅ 加入 V7 的上帝视角代码更新完毕！")

📦 [1/4] 加载模型数据 (V1 -> V6 -> V7 -> V8)...
🧩 [2/4] 构建特征与排位库...
⚔️ [3/4] 圈定分析人群...
📊 [4/4] 生成终极对账单...

🎯 【冲突区 A】 V1 假阳性陷阱 (V1爱, C恨) | 样本量: 42845
[Uplift 全局排位演进 (数值越小越头部)]
  盲人底座 (V1) :   6.60%
  残差防守 (V6) :   7.81%
  全局衰减 (V7) :   5.87%
  特征门控 (S4) :   5.65%
  动态阈值 (S1) :   6.48%

[门控流向解剖 (上帝先验 -> V7衰减 -> V8重配)]
  🔵 C-Prior先验:   [CO:0.000, AT:0.202, NT:0.803]  ==> 占比: (CO:  0.0%, AT: 20.1%, NT: 79.9%)
  🟡 V7 全局衰减:   [CO:0.000, AT:0.000, NT:0.000]  ==> 占比: (CO:  0.0%, AT: 20.1%, NT: 79.9%) | 统一 Gate: nan
  🟢 V8_S4纯门控:   [CO:0.522, AT:0.818, NT:0.260]  <-- 网络独立计算方向
  🟢 V8_S4最终Pi:   [CO:0.000, AT:0.166, NT:0.210]  ==> 占比: (CO:  0.0%, AT: 44.2%, NT: 55.8%)
  🟠 V8_S1纯门控:   [CO:1.000, AT:1.000, NT:1.000]  <-- 网络平移阈值
  🟠 V8_S1最终Pi:   [CO:0.000, AT:0.202, NT:0.803]  ==> 占比: (CO:  0.0%, AT: 20.1%, NT: 79.9%) | Delta: +nan
------------------------------------------------------------------------------------------------------------------------

🎯 【冲突区 B】 C 遗漏的金子 (C爱, V1恨) | 样本量: 17384
[Uplift 全局排位演进 

In [1]:
import numpy as np
import os

files_to_check = [
    "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s4/run_v8_s4/train_dist_embeddings.npz",
    "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s4/run_v8_s4/valid_dist_embeddings.npz",
    "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s4/run_v8_s4/test_dist_embeddings.npz"
]

print("="*80)
print(f"🚀 V8_S4 隐藏状态 (Embeddings) 暴力体检")
print("="*80)

for file_path in files_to_check:
    print(f"\n📁 正在检查: {os.path.basename(file_path)}")
    if not os.path.exists(file_path):
        print(f"  ❌ 文件不存在，请检查路径！")
        continue
        
    try:
        data = np.load(file_path)
        keys = data.files
        print(f"  ✅ 成功加载！包含 {len(keys)} 个数组 (Keys): {keys}")
        
        print("-" * 60)
        print(f"  {'Array Name':<20} | {'Shape':<15} | {'NaN数':<6} | {'Min':>8} | {'Max':>8} | {'Mean':>8}")
        print("-" * 60)
        
        for key in keys:
            arr = data[key]
            
            # 如果不是数值型数组（比如存了字符串），跳过统计
            if not np.issubdtype(arr.dtype, np.number):
                print(f"  {key:<20} | {str(arr.shape):<15} | {'N/A':<6} | {'N/A':>8} | {'N/A':>8} | {'N/A':>8}")
                continue
                
            nan_count = np.isnan(arr).sum()
            inf_count = np.isinf(arr).sum()
            
            # 处理全 NaN 或全 Inf 的情况，防止算 mean 报错
            valid_arr = arr[~np.isnan(arr) & ~np.isinf(arr)]
            
            if len(valid_arr) == 0:
                print(f"  {key:<20} | {str(arr.shape):<15} | {nan_count:<6} | {'ALL BAD':>8} | {'ALL BAD':>8} | {'ALL BAD':>8}")
            else:
                a_min = valid_arr.min()
                a_max = valid_arr.max()
                a_mean = valid_arr.mean()
                
                # 如果有 Inf 也要标出来
                nan_str = f"{nan_count}" + (f"(+{inf_count}inf)" if inf_count > 0 else "")
                print(f"  {key:<20} | {str(arr.shape):<15} | {nan_str:<6} | {a_min:>8.3f} | {a_max:>8.3f} | {a_mean:>8.3f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8_S4 隐藏状态 (Embeddings) 暴力体检

📁 正在检查: train_dist_embeddings.npz
  ✅ 成功加载！包含 1 个数组 (Keys): ['wb_shared_emb']
------------------------------------------------------------
  Array Name           | Shape           | NaN数   |      Min |      Max |     Mean
------------------------------------------------------------
  wb_shared_emb        | (11183673, 32)  | 0      |    0.000 |   18.631 |    2.134

📁 正在检查: valid_dist_embeddings.npz
  ✅ 成功加载！包含 1 个数组 (Keys): ['wb_shared_emb']
------------------------------------------------------------
  Array Name           | Shape           | NaN数   |      Min |      Max |     Mean
------------------------------------------------------------
  wb_shared_emb        | (1397959, 32)   | 0      |    0.000 |   18.538 |    2.133

📁 正在检查: test_dist_embeddings.npz
  ✅ 成功加载！包含 1 个数组 (Keys): ['wb_shared_emb']
------------------------------------------------------------
  Array Name           | Shape           | NaN数   |      Min |      Max |     Mean
-------------

In [4]:
import pandas as pd
import os

# 👉 修改为你真正想排查的模型目录 (猜你想看 s1_t10)
base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s1_t10/run_v8_s1_t10"

files = {
    "【1】Train (训练集)": os.path.join(base_dir, "train_dist.csv"),
    "【2】Valid (验证集)": os.path.join(base_dir, "valid_dist.csv"),
    "【3】Test  (测试集)": os.path.join(base_dir, "test_dist.csv")
}

# 我们要盯死的嫌疑犯列
cols_to_check = [
    'wb_delta_c', 
    'wb_gate_comp', 'wb_gate_always', 'wb_gate_never',
    'wb_final_pi_comp', 'wb_final_pi_always', 'wb_final_pi_never',
    'uplift_pred'
]

print("="*95)
print("🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单")
print("="*95)

for split_name, path in files.items():
    print(f"\n📊 正在分析 {split_name} -> {os.path.basename(path)}")
    if not os.path.exists(path):
        print("  ❌ 文件不存在！(可能还没跑完预测脚本)")
        continue
        
    try:
        df = pd.read_csv(path)
        existing_cols = [c for c in cols_to_check if c in df.columns]
        missing_cols = [c for c in cols_to_check if c not in df.columns]
        
        if missing_cols:
            print(f"  ⚠️ 提示: 缺失列 {missing_cols} (可能是之前的代码没把它们加进 pi_dict)")
            
        print("-" * 90)
        print(f"  {'字段名 (Column)':<20} | {'NaN数':<8} | {'Min':>8} | {'Max':>8} | {'Mean':>8} | {'Std (波动)':>10}")
        print("-" * 90)
        
        for col in existing_cols:
            s = df[col]
            nan_count = s.isna().sum()
            if nan_count == len(s):
                print(f"  {col:<20} | {nan_count:<8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>10}")
            else:
                print(f"  {col:<20} | {nan_count:<8} | {s.min():>8.4f} | {s.max():>8.4f} | {s.mean():>8.4f} | {s.std():>10.4f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单

📊 正在分析 【1】Train (训练集) -> train_dist.csv
  ⚠️ 提示: 缺失列 ['wb_delta_c'] (可能是之前的代码没把它们加进 pi_dict)
------------------------------------------------------------------------------------------
  字段名 (Column)         | NaN数     |      Min |      Max |     Mean |   Std (波动)
------------------------------------------------------------------------------------------
  wb_gate_comp         | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_always       | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_never        | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_final_pi_comp     | 0        |   0.0000 |   0.3533 |   0.0077 |     0.0228
  wb_final_pi_always   | 0        |   0.0016 |   0.9254 |   0.0408 |     0.0974
  wb_final_pi_never    | 0        |   0.0391 |   0.9987 |   0.9519 |     0.1133
  uplift_pred          | 0        |   0.0001 |   0.1187 |   0.0021 |     0.0047

📊 正在分析 【2】Valid (验证集) -> valid_dist.csv

In [5]:
import pandas as pd
import os

# 👉 修改为你真正想排查的模型目录 (猜你想看 s1_t10)
base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s1_t5/run_v8_s1_t5"

files = {
    "【1】Train (训练集)": os.path.join(base_dir, "train_dist.csv"),
    "【2】Valid (验证集)": os.path.join(base_dir, "valid_dist.csv"),
    "【3】Test  (测试集)": os.path.join(base_dir, "test_dist.csv")
}

# 我们要盯死的嫌疑犯列
cols_to_check = [
    'wb_delta_c', 
    'wb_gate_comp', 'wb_gate_always', 'wb_gate_never',
    'wb_final_pi_comp', 'wb_final_pi_always', 'wb_final_pi_never',
    'uplift_pred'
]

print("="*95)
print("🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单")
print("="*95)

for split_name, path in files.items():
    print(f"\n📊 正在分析 {split_name} -> {os.path.basename(path)}")
    if not os.path.exists(path):
        print("  ❌ 文件不存在！(可能还没跑完预测脚本)")
        continue
        
    try:
        df = pd.read_csv(path)
        existing_cols = [c for c in cols_to_check if c in df.columns]
        missing_cols = [c for c in cols_to_check if c not in df.columns]
        
        if missing_cols:
            print(f"  ⚠️ 提示: 缺失列 {missing_cols} (可能是之前的代码没把它们加进 pi_dict)")
            
        print("-" * 90)
        print(f"  {'字段名 (Column)':<20} | {'NaN数':<8} | {'Min':>8} | {'Max':>8} | {'Mean':>8} | {'Std (波动)':>10}")
        print("-" * 90)
        
        for col in existing_cols:
            s = df[col]
            nan_count = s.isna().sum()
            if nan_count == len(s):
                print(f"  {col:<20} | {nan_count:<8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>10}")
            else:
                print(f"  {col:<20} | {nan_count:<8} | {s.min():>8.4f} | {s.max():>8.4f} | {s.mean():>8.4f} | {s.std():>10.4f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单

📊 正在分析 【1】Train (训练集) -> train_dist.csv
  ⚠️ 提示: 缺失列 ['wb_delta_c'] (可能是之前的代码没把它们加进 pi_dict)
------------------------------------------------------------------------------------------
  字段名 (Column)         | NaN数     |      Min |      Max |     Mean |   Std (波动)
------------------------------------------------------------------------------------------
  wb_gate_comp         | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_always       | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_never        | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_final_pi_comp     | 0        |   0.0000 |   0.3533 |   0.0077 |     0.0228
  wb_final_pi_always   | 0        |   0.0016 |   0.9254 |   0.0408 |     0.0974
  wb_final_pi_never    | 0        |   0.0391 |   0.9987 |   0.9519 |     0.1133
  uplift_pred          | 0        |  -0.0016 |   0.5288 |   0.0021 |     0.0127

📊 正在分析 【2】Valid (验证集) -> valid_dist.csv

In [6]:
import pandas as pd
import os

# 👉 修改为你真正想排查的模型目录 (猜你想看 s1_t10)
base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s1_t50/run_v8_s1_t50"

files = {
    "【1】Train (训练集)": os.path.join(base_dir, "train_dist.csv"),
    "【2】Valid (验证集)": os.path.join(base_dir, "valid_dist.csv"),
    "【3】Test  (测试集)": os.path.join(base_dir, "test_dist.csv")
}

# 我们要盯死的嫌疑犯列
cols_to_check = [
    'wb_delta_c', 
    'wb_gate_comp', 'wb_gate_always', 'wb_gate_never',
    'wb_final_pi_comp', 'wb_final_pi_always', 'wb_final_pi_never',
    'uplift_pred'
]

print("="*95)
print("🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单")
print("="*95)

for split_name, path in files.items():
    print(f"\n📊 正在分析 {split_name} -> {os.path.basename(path)}")
    if not os.path.exists(path):
        print("  ❌ 文件不存在！(可能还没跑完预测脚本)")
        continue
        
    try:
        df = pd.read_csv(path)
        existing_cols = [c for c in cols_to_check if c in df.columns]
        missing_cols = [c for c in cols_to_check if c not in df.columns]
        
        if missing_cols:
            print(f"  ⚠️ 提示: 缺失列 {missing_cols} (可能是之前的代码没把它们加进 pi_dict)")
            
        print("-" * 90)
        print(f"  {'字段名 (Column)':<20} | {'NaN数':<8} | {'Min':>8} | {'Max':>8} | {'Mean':>8} | {'Std (波动)':>10}")
        print("-" * 90)
        
        for col in existing_cols:
            s = df[col]
            nan_count = s.isna().sum()
            if nan_count == len(s):
                print(f"  {col:<20} | {nan_count:<8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>10}")
            else:
                print(f"  {col:<20} | {nan_count:<8} | {s.min():>8.4f} | {s.max():>8.4f} | {s.mean():>8.4f} | {s.std():>10.4f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单

📊 正在分析 【1】Train (训练集) -> train_dist.csv
  ⚠️ 提示: 缺失列 ['wb_delta_c'] (可能是之前的代码没把它们加进 pi_dict)
------------------------------------------------------------------------------------------
  字段名 (Column)         | NaN数     |      Min |      Max |     Mean |   Std (波动)
------------------------------------------------------------------------------------------
  wb_gate_comp         | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_always       | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_gate_never        | 0        |   1.0000 |   1.0000 |   1.0000 |     0.0000
  wb_final_pi_comp     | 0        |   0.0000 |   0.3533 |   0.0077 |     0.0228
  wb_final_pi_always   | 0        |   0.0016 |   0.9254 |   0.0408 |     0.0974
  wb_final_pi_never    | 0        |   0.0391 |   0.9987 |   0.9519 |     0.1133
  uplift_pred          | 0        |  -0.0115 |   0.3581 |   0.0009 |     0.0079

📊 正在分析 【2】Valid (验证集) -> valid_dist.csv

In [7]:
import pandas as pd
import os

# 👉 修改为你真正想排查的模型目录 (猜你想看 s1_t10)
base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s2_t10/run_v8_s2_t10"

files = {
    "【1】Train (训练集)": os.path.join(base_dir, "train_dist.csv"),
    "【2】Valid (验证集)": os.path.join(base_dir, "valid_dist.csv"),
    "【3】Test  (测试集)": os.path.join(base_dir, "test_dist.csv")
}

# 我们要盯死的嫌疑犯列
cols_to_check = [
    'wb_delta_c', 
    'wb_gate_comp', 'wb_gate_always', 'wb_gate_never',
    'wb_final_pi_comp', 'wb_final_pi_always', 'wb_final_pi_never',
    'uplift_pred'
]

print("="*95)
print("🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单")
print("="*95)

for split_name, path in files.items():
    print(f"\n📊 正在分析 {split_name} -> {os.path.basename(path)}")
    if not os.path.exists(path):
        print("  ❌ 文件不存在！(可能还没跑完预测脚本)")
        continue
        
    try:
        df = pd.read_csv(path)
        existing_cols = [c for c in cols_to_check if c in df.columns]
        missing_cols = [c for c in cols_to_check if c not in df.columns]
        
        if missing_cols:
            print(f"  ⚠️ 提示: 缺失列 {missing_cols} (可能是之前的代码没把它们加进 pi_dict)")
            
        print("-" * 90)
        print(f"  {'字段名 (Column)':<20} | {'NaN数':<8} | {'Min':>8} | {'Max':>8} | {'Mean':>8} | {'Std (波动)':>10}")
        print("-" * 90)
        
        for col in existing_cols:
            s = df[col]
            nan_count = s.isna().sum()
            if nan_count == len(s):
                print(f"  {col:<20} | {nan_count:<8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>10}")
            else:
                print(f"  {col:<20} | {nan_count:<8} | {s.min():>8.4f} | {s.max():>8.4f} | {s.mean():>8.4f} | {s.std():>10.4f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单

📊 正在分析 【1】Train (训练集) -> train_dist.csv
  ⚠️ 提示: 缺失列 ['wb_delta_c'] (可能是之前的代码没把它们加进 pi_dict)
------------------------------------------------------------------------------------------
  字段名 (Column)         | NaN数     |      Min |      Max |     Mean |   Std (波动)
------------------------------------------------------------------------------------------
  wb_gate_comp         | 0        |   0.0001 |   1.0000 |   0.1038 |     0.2944
  wb_gate_always       | 0        |   0.0001 |   1.0000 |   0.1038 |     0.2944
  wb_gate_never        | 0        |   0.0001 |   1.0000 |   0.1038 |     0.2944
  wb_final_pi_comp     | 0        |   0.0000 |   0.3533 |   0.0064 |     0.0229
  wb_final_pi_always   | 0        |   0.0000 |   0.9247 |   0.0225 |     0.0855
  wb_final_pi_never    | 0        |   0.0001 |   0.9496 |   0.0749 |     0.2177
  uplift_pred          | 0        |  -0.0040 |   0.1716 |  -0.0003 |     0.0059

📊 正在分析 【2】Valid (验证集) -> valid_dist.csv

In [8]:
import pandas as pd
import os

# 👉 修改为你真正想排查的模型目录 (猜你想看 s1_t10)
base_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/y_v8_s3_t10/run_v8_s3_t10"

files = {
    "【1】Train (训练集)": os.path.join(base_dir, "train_dist.csv"),
    "【2】Valid (验证集)": os.path.join(base_dir, "valid_dist.csv"),
    "【3】Test  (测试集)": os.path.join(base_dir, "test_dist.csv")
}

# 我们要盯死的嫌疑犯列
cols_to_check = [
    'wb_delta_c', 
    'wb_gate_comp', 'wb_gate_always', 'wb_gate_never',
    'wb_final_pi_comp', 'wb_final_pi_always', 'wb_final_pi_never',
    'uplift_pred'
]

print("="*95)
print("🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单")
print("="*95)

for split_name, path in files.items():
    print(f"\n📊 正在分析 {split_name} -> {os.path.basename(path)}")
    if not os.path.exists(path):
        print("  ❌ 文件不存在！(可能还没跑完预测脚本)")
        continue
        
    try:
        df = pd.read_csv(path)
        existing_cols = [c for c in cols_to_check if c in df.columns]
        missing_cols = [c for c in cols_to_check if c not in df.columns]
        
        if missing_cols:
            print(f"  ⚠️ 提示: 缺失列 {missing_cols} (可能是之前的代码没把它们加进 pi_dict)")
            
        print("-" * 90)
        print(f"  {'字段名 (Column)':<20} | {'NaN数':<8} | {'Min':>8} | {'Max':>8} | {'Mean':>8} | {'Std (波动)':>10}")
        print("-" * 90)
        
        for col in existing_cols:
            s = df[col]
            nan_count = s.isna().sum()
            if nan_count == len(s):
                print(f"  {col:<20} | {nan_count:<8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>8} | {'ALL NaN':>10}")
            else:
                print(f"  {col:<20} | {nan_count:<8} | {s.min():>8.4f} | {s.max():>8.4f} | {s.mean():>8.4f} | {s.std():>10.4f}")
                
    except Exception as e:
        print(f"  ❌ 读取崩溃: {e}")

🚀 V8 核心变量跨阶段 (Train vs Valid vs Test) 体检对账单

📊 正在分析 【1】Train (训练集) -> train_dist.csv
  ⚠️ 提示: 缺失列 ['wb_delta_c'] (可能是之前的代码没把它们加进 pi_dict)
------------------------------------------------------------------------------------------
  字段名 (Column)         | NaN数     |      Min |      Max |     Mean |   Std (波动)
------------------------------------------------------------------------------------------
  wb_gate_comp         | 0        |   0.0002 |   1.0000 |   0.1062 |     0.2972
  wb_gate_always       | 0        |   0.0000 |   1.0000 |   0.0984 |     0.2878
  wb_gate_never        | 0        |   0.0000 |   1.0000 |   0.0980 |     0.2874
  wb_final_pi_comp     | 0        |   0.0000 |   0.3533 |   0.0064 |     0.0229
  wb_final_pi_always   | 0        |   0.0000 |   0.9221 |   0.0217 |     0.0843
  wb_final_pi_never    | 0        |   0.0000 |   0.9455 |   0.0700 |     0.2108
  uplift_pred          | 0        |  -0.0815 |  -0.0002 |  -0.0476 |     0.0135

📊 正在分析 【2】Valid (验证集) -> valid_dist.csv

: 